# NYC Taxi | January 2025 raw ingestion and Silver preparation

## Goal
Read externally sourced TLC files, preserve January records, enrich locations, flag measurement suitability, and persist a Silver Delta table.

## Setup
Upload these two files into `/Volumes/workspace/default/nyc_taxi_raw/` before running:
- `yellow_tripdata_2025-01.parquet`
- `taxi_zone_lookup.csv`

See [the TLC source page](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page). The raw files are the Bronze input; this notebook does **not** create a separate Bronze Delta table. The historical filename is retained.

The project counts records, not unique passengers. All January records remain in Silver. Only duration, distance, and speed define measurement eligibility; fares are not an eligibility condition. This pipeline uses `F.timestamp_diff`, available in PySpark 4.0+.

**Write behavior:** the final `CREATE OR REPLACE TABLE` creates or fully refreshes `workspace.default.nyc_taxi_silver_2025_01`. It does not append duplicates. Selected unchanged cells retain recorded outputs; edited cells must be rerun in Databricks.

## Steps

In [ ]:
from pyspark.sql import functions as F

raw_volume = "/Volumes/workspace/default/nyc_taxi_raw"
assert hasattr(F, "timestamp_diff"), "Use a Databricks Python environment with PySpark 4.0+."

### Load the raw January file

In [ ]:
taxi_raw = spark.read.parquet(
    f"{raw_volume}/yellow_tripdata_2025-01.parquet"
)

taxi_raw.printSchema()
display(taxi_raw.limit(5))

### Inspect payment categories

In [ ]:
display(
    taxi_raw
    .groupBy("payment_type")
    .count()
    .orderBy("payment_type")
)

payment_type,count
0,540149
1,2444393
2,390429
3,23773
4,76481
5,1


### Inspect rate codes

In [ ]:
display(
    taxi_raw
    .groupBy("RatecodeID")
    .count()
    .orderBy("RatecodeID")
)

RatecodeID,count
null,540149
1,2756472
2,94420
3,8622
4,7092
5,26501
6,7
99,41963


### Inspect missing rate codes within payment category 0

In [ ]:
display(
    taxi_raw
    .filter("payment_type = 0")
    .groupBy("RatecodeID")
    .count()
    .orderBy("RatecodeID")
)

RatecodeID,count
null,540149


**Recorded observation:** 540,149 raw-file records have payment category 0 and a null `RatecodeID`. The TLC dictionary identifies category 0 as Flex Fare. Missing rate codes alone are not used to discard these records.

### Check raw row count and date range

In [ ]:
from pyspark.sql import functions as F

display(
    taxi_raw.agg(
        F.count("*").alias("kayit_sayisi"),
        F.min("tpep_pickup_datetime").alias("ilk_baslangic"),
        F.max("tpep_pickup_datetime").alias("son_baslangic")
    )
)

kayit_sayisi,ilk_baslangic,son_baslangic
3475226,2024-12-31T20:47:55.000,2025-02-01T00:00:44.000


### Count records inside and outside January

In [ ]:
baslangic = F.col("tpep_pickup_datetime")

display(
    taxi_raw
    .withColumn(
        "tarih_grubu",
        F.when(baslangic.isNull(), "Boş")
         .when(baslangic < "2025-01-01", "Ocak öncesi")
         .when(baslangic >= "2025-02-01", "Ocak sonrası")
         .otherwise("Ocak içi")
    )
    .groupBy("tarih_grubu")
    .count()
)

tarih_grubu,count
Ocak öncesi,21
Ocak sonrası,1
Ocak içi,3475204


### Select January pickup timestamps

In [ ]:
taxi_jan = taxi_raw.filter(
    (F.col("tpep_pickup_datetime") >= "2025-01-01") &
    (F.col("tpep_pickup_datetime") < "2025-02-01")
)

print(taxi_jan.count())

3475204


### Calculate elapsed duration

In [ ]:
taxi_sure = taxi_jan.withColumn(
    "sure_dakika",
    F.timestamp_diff(
        "SECOND",
        F.col("tpep_pickup_datetime"),
        F.col("tpep_dropoff_datetime")
    ) / 60.0
)

display(
    taxi_sure.select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "sure_dakika",
        "trip_distance"
    ).limit(5)
)

tpep_pickup_datetime,tpep_dropoff_datetime,sure_dakika,trip_distance
2025-01-01T00:18:38.000,2025-01-01T00:26:59.000,8.35,1.6
2025-01-01T00:32:40.000,2025-01-01T00:35:13.000,2.55,0.5
2025-01-01T00:44:04.000,2025-01-01T00:46:01.000,1.95,0.6
2025-01-01T00:14:27.000,2025-01-01T00:20:01.000,5.566666666666666,0.52
2025-01-01T00:21:34.000,2025-01-01T00:25:06.000,3.533333333333333,0.66


### Review duration groups

In [ ]:
sure = F.col("sure_dakika")

display(
    taxi_sure
    .withColumn(
        "sure_grubu",
        F.when(sure.isNull(), "Boş")
         .when(sure <= 0, "Sıfır veya negatif")
         .when(sure > 1200, "20 saatten uzun")
         .when(sure > 180, "3–20 saat")
         .otherwise("0–3 saat")
    )
    .groupBy("sure_grubu")
    .count()
)

sure_grubu,count
3–20 saat,442
0–3 saat,3471776
Sıfır veya negatif,2051
20 saatten uzun,935


### Inspect the longest durations

In [ ]:
display(
    taxi_sure
    .filter(F.col("sure_dakika") > 1200)
    .select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "sure_dakika",
        "trip_distance",
        "fare_amount",
        "payment_type"
    )
    .orderBy(F.col("sure_dakika").desc())
    .limit(10)
)

tpep_pickup_datetime,tpep_dropoff_datetime,sure_dakika,trip_distance,fare_amount,payment_type
2025-01-21T12:23:41.000,2025-01-25T10:10:00.000,5626.316666666667,6.55,28.2,2
2025-01-21T12:23:41.000,2025-01-25T10:10:00.000,5626.316666666667,6.55,-28.2,2
2025-01-06T02:12:11.000,2025-01-09T13:54:19.000,5022.133333333333,91.51,336.9,2
2025-01-27T11:58:55.000,2025-01-30T09:42:55.000,4184.0,0.0,3.0,2
2025-01-11T08:04:00.000,2025-01-13T16:49:28.000,3405.4666666666667,0.04,70.0,2
2025-01-24T07:07:21.000,2025-01-26T10:09:44.000,3062.383333333333,0.0,3.0,2
2025-01-02T09:40:31.000,2025-01-04T09:37:01.000,2876.5,0.0,3.0,2
2025-01-17T16:45:49.000,2025-01-19T16:40:59.000,2875.1666666666665,1.98,12.8,2
2025-01-07T18:07:33.000,2025-01-09T16:12:05.000,2764.5333333333333,1.86,47.8,2
2025-01-29T21:39:23.000,2025-01-31T16:48:43.000,2589.3333333333335,7.52,31.02,0


**Recorded observation:** 935 January records exceed 20 hours and require investigation before duration analysis.

### Load taxi-zone reference data

In [ ]:
zones = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{raw_volume}/taxi_zone_lookup.csv")
)

zones.printSchema()
display(zones.limit(5))

### Check lookup keys

In [ ]:
zones.select(
    F.count("*").alias("toplam_satir"),
    F.countDistinct("LocationID").alias("benzersiz_location_id")
).display()

toplam_satir,benzersiz_location_id
265,265


In [ ]:
zone_stats = zones.agg(
    F.count("*").alias("rows"),
    F.countDistinct("LocationID").alias("unique_ids"),
    F.sum(F.col("LocationID").isNull().cast("int")).alias("null_ids")
).first()
assert zone_stats["rows"] == zone_stats["unique_ids"] and zone_stats["null_ids"] == 0,     "Taxi-zone lookup must contain one non-null row per LocationID."

### Enrich pickup location

In [ ]:
pickup_zones = zones.select(
    F.col("LocationID").alias("PULocationID"),
    F.col("Borough").alias("pickup_borough"),
    F.col("Zone").alias("pickup_zone")
)

taxi_pickup = taxi_sure.join(
    pickup_zones,
    on="PULocationID",
    how="left"
)

display(
    taxi_pickup.select(
        "PULocationID", "pickup_borough", "pickup_zone", "sure_dakika"
    ).limit(10)
)

PULocationID,pickup_borough,pickup_zone,sure_dakika
229,Manhattan,Sutton Place/Turtle Bay North,8.35
236,Manhattan,Upper East Side North,2.55
141,Manhattan,Lenox Hill West,1.95
244,Manhattan,Washington Heights South,5.566666666666666
244,Manhattan,Washington Heights South,3.533333333333333
239,Manhattan,Upper West Side South,20.033333333333335
170,Manhattan,Murray Hill,1.4666666666666666
234,Manhattan,Union Sq,12.4
148,Manhattan,Lower East Side,19.666666666666668
237,Manhattan,Upper East Side South,9.566666666666666


### Enrich drop-off location

In [ ]:
dropoff_zones = zones.select(
    F.col("LocationID").alias("DOLocationID"),
    F.col("Borough").alias("dropoff_borough"),
    F.col("Zone").alias("dropoff_zone")
)

taxi_bolgeler = taxi_pickup.join(
    dropoff_zones,
    on="DOLocationID",
    how="left"
)

display(
    taxi_bolgeler.select(
        "pickup_zone", "dropoff_zone", "sure_dakika"
    ).limit(10)
)

pickup_zone,dropoff_zone,sure_dakika
Sutton Place/Turtle Bay North,Upper East Side South,8.35
Upper East Side North,Upper East Side South,2.55
Lenox Hill West,Lenox Hill West,1.95
Washington Heights South,Washington Heights South,5.566666666666666
Washington Heights South,Hamilton Heights,3.533333333333333
Upper West Side South,East Chelsea,20.033333333333335
Murray Hill,Murray Hill,1.4666666666666666
Union Sq,Lower East Side,12.4
Lower East Side,Murray Hill,19.666666666666668
Upper East Side South,Yorkville East,9.566666666666666


### Reconcile join count and unmatched locations

In [ ]:
taxi_bolgeler.select(
    F.count("*").alias("yolculuk_sayisi"),
    F.sum(F.col("pickup_zone").isNull().cast("int")).alias("binis_bolgesi_yok"),
    F.sum(F.col("dropoff_zone").isNull().cast("int")).alias("inis_bolgesi_yok")
).display()

yolculuk_sayisi,binis_bolgesi_yok,inis_bolgesi_yok
3475204,0,0


In [ ]:
assert taxi_bolgeler.count() == taxi_jan.count(),     "A zone join multiplied the January records; inspect lookup keys."

### Rank pickup zones

In [ ]:
display(
    taxi_bolgeler
    .groupBy("pickup_borough", "pickup_zone")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
)

pickup_borough,pickup_zone,count
Manhattan,Midtown Center,169977
Manhattan,Upper East Side South,163703
Manhattan,Upper East Side North,155647
Queens,JFK Airport,146135
Manhattan,Times Sq/Theatre District,125829
Manhattan,Penn Station/Madison Sq West,119130
Manhattan,Midtown East,117930
Manhattan,Lincoln Square East,110584
Manhattan,Upper West Side South,96614
Manhattan,Midtown North,95905


### Rank pickup–drop-off pairs

In [ ]:
display(
    taxi_bolgeler
    .groupBy("pickup_zone", "dropoff_zone")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
)

pickup_zone,dropoff_zone,count
Upper East Side South,Upper East Side North,24839
Upper East Side North,Upper East Side South,21843
Upper East Side North,Upper East Side North,18221
Upper East Side South,Upper East Side South,17441
Midtown Center,Upper East Side South,11463
Upper East Side South,Midtown Center,10168
Midtown Center,Upper East Side North,9845
Upper West Side South,Upper West Side North,9595
Lincoln Square East,Upper West Side South,9141
Upper West Side South,Lincoln Square East,8844


### Inspect distance quality

In [ ]:
taxi_bolgeler.select(
    F.count("*").alias("toplam"),
    F.sum(F.col("trip_distance").isNull().cast("int")).alias("mesafe_bos"),
    F.sum((F.col("trip_distance") <= 0).cast("int")).alias("mesafe_sifir_veya_negatif"),
    F.min("trip_distance").alias("en_kucuk_mesafe"),
    F.max("trip_distance").alias("en_buyuk_mesafe")
).display()

toplam,mesafe_bos,mesafe_sifir_veya_negatif,en_kucuk_mesafe,en_buyuk_mesafe
3475204,0,90893,0.0,276423.57


### Inspect the longest distances

In [ ]:
display(
    taxi_bolgeler
    .select(
        "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "pickup_zone", "dropoff_zone",
        "trip_distance", "sure_dakika", "total_amount"
    )
    .orderBy(F.desc("trip_distance"))
    .limit(10)
)

tpep_pickup_datetime,tpep_dropoff_datetime,pickup_zone,dropoff_zone,trip_distance,sure_dakika,total_amount
2025-01-22T21:04:00.000,2025-01-22T21:13:00.000,Midtown Center,Murray Hill,276423.57,9.0,5.0
2025-01-19T14:51:00.000,2025-01-19T14:58:00.000,Stuy Town/Peter Cooper Village,UN/Turtle Bay South,276099.95,7.0,13.88
2025-01-21T09:19:00.000,2025-01-21T09:45:00.000,Cobble Hill,Little Italy/NoLiTa,222167.49,26.0,35.94
2025-01-16T10:47:00.000,2025-01-16T11:11:00.000,Yorkville West,Clinton East,206137.99,24.0,29.64
2025-01-26T17:46:00.000,2025-01-26T17:59:00.000,Clinton West,Penn Station/Madison Sq West,202771.63,13.0,14.85
2025-01-14T19:18:00.000,2025-01-14T19:36:00.000,West Chelsea/Hudson Yards,Midtown South,189687.43,18.0,17.45
2025-01-13T21:11:00.000,2025-01-13T21:17:00.000,Garment District,Flatiron,181139.99,6.0,11.08
2025-01-24T08:33:00.000,2025-01-24T08:44:00.000,Central Harlem North,Upper West Side North,168079.57,11.0,4.49
2025-01-19T18:31:00.000,2025-01-19T18:58:00.000,West Concourse,Upper West Side South,167452.94,27.0,5.45
2025-01-17T14:25:00.000,2025-01-17T14:40:00.000,Yorkville East,Upper West Side South,164959.95,15.0,18.05


### Count distance outliers

In [ ]:
taxi_bolgeler.select(
    F.sum((F.col("trip_distance") > 100).cast("int")).alias("100_mil_ustu"),
    F.sum((F.col("trip_distance") > 1000).cast("int")).alias("1000_mil_ustu")
).display()

100_mil_ustu,1000_mil_ustu
162,116


### Label distance quality

In [ ]:
taxi_kalite = taxi_bolgeler.withColumn(
    "mesafe_durumu",
    F.when(F.col("trip_distance").isNull(), "Boş mesafe")
     .when(F.col("trip_distance") < 0, "Negatif mesafe")
     .when(F.col("trip_distance") == 0, "Sıfır mesafe")
     .when(F.col("trip_distance") > 1000, "Aşırı uç - incele")
     .when(F.col("trip_distance") > 100, "Uzun - incele")
     .otherwise("0-100 mil")
)

display(taxi_kalite.groupBy("mesafe_durumu").count())

### Label duration quality

In [ ]:
taxi_kalite = taxi_kalite.withColumn(
    "sure_durumu",
    F.when(F.col("sure_dakika").isNull(), "Boş süre")
     .when(F.col("sure_dakika") <= 0, "Sıfır veya negatif")
     .when(F.col("sure_dakika") > 1200, "20 saatten uzun - incele")
     .when(F.col("sure_dakika") > 180, "3-20 saat - incele")
     .otherwise("0-3 saat")
)

display(taxi_kalite.groupBy("sure_durumu").count())

sure_durumu,count
20 saatten uzun - incele,935
0-3 saat,3471776
3-20 saat - incele,442
Sıfır veya negatif,2051


### Cross-tabulate distance and duration labels

In [ ]:
display(
    taxi_kalite
    .groupBy("sure_durumu")
    .pivot("mesafe_durumu", [
        "0-100 mil",
        "Boş mesafe",
        "Negatif mesafe",
        "Sıfır mesafe",
        "Uzun - incele",
        "Aşırı uç - incele"
    ])
    .count()
    .fillna(0)
)

### Review zero-distance records by payment category

In [ ]:
display(
    taxi_kalite
    .filter(F.col("trip_distance") == 0)
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("kayit"),
        F.sum((F.col("total_amount") > 0).cast("int")).alias("pozitif_tutarli")
    )
    .orderBy(F.desc("kayit"))
)

payment_type,kayit,pozitif_tutarli
0,51983,51941
1,19649,19609
2,8926,7984
4,5495,3234
3,4839,3337
5,1,0


### Select distance and duration candidates

In [ ]:
taxi_olcum = taxi_kalite.filter(
    (F.col("trip_distance") > 0) &
    (F.col("trip_distance") <= 100) &
    (F.col("sure_dakika") > 0) &
    (F.col("sure_dakika") <= 180)
)

print("Ölçüm analizine aday kayıt:", taxi_olcum.count())

### Calculate speed and inspect outliers

In [ ]:
taxi_olcum = taxi_olcum.withColumn(
    "ortalama_hiz_mph",
    F.col("trip_distance") * 60 / F.col("sure_dakika")
)

display(
    taxi_olcum.select(
        F.sum((F.col("ortalama_hiz_mph") > 80).cast("int")).alias("80_mph_ustu"),
        F.max("ortalama_hiz_mph").alias("en_yuksek_hiz")
    )
)

80_mph_ustu,en_yuksek_hiz
857,67608.00000000001


### Label speed candidates

In [ ]:
taxi_olcum = taxi_olcum.withColumn(
    "hiz_durumu",
    F.when(
        F.col("ortalama_hiz_mph") > 80,
        "Yüksek hız - incele"
    ).otherwise("80 mph ve altı")
)

display(taxi_olcum.groupBy("hiz_durumu").count())

hiz_durumu,count
80 mph ve altı,3380592
Yüksek hız - incele,857


### Inspect implausible speeds

In [ ]:
display(
    taxi_olcum
    .filter(F.col("ortalama_hiz_mph") > 80)
    .select(
        "pickup_zone",
        "dropoff_zone",
        "trip_distance",
        "sure_dakika",
        "ortalama_hiz_mph",
        "total_amount"
    )
    .orderBy(F.desc("ortalama_hiz_mph"))
    .limit(10)
)

pickup_zone,dropoff_zone,trip_distance,sure_dakika,ortalama_hiz_mph,total_amount
Rosedale,Central Harlem North,18.78,0.016666666666666666,67608.00000000001,60.2
Baisley Park,Upper West Side North,16.27,0.016666666666666666,58572.0,53.05
Flushing,Lincoln Square East,13.22,0.016666666666666666,47592.0,39.15
Times Sq/Theatre District,Times Sq/Theatre District,8.12,0.016666666666666666,29231.999999999996,51.02
East Harlem North,East Harlem North,7.12,0.016666666666666666,25632.0,37.67
Steinway,South Jamaica,10.9,0.03333333333333333,19620.0,31.68
South Ozone Park,South Ozone Park,13.8,0.05,16560.0,8.75
Garment District,Garment District,17.6,0.06666666666666667,15840.0,74.75
Kips Bay,Crown Heights North,8.65,0.03333333333333333,15570.0,33.3
Cobble Hill,UN/Turtle Bay South,4.2,0.016666666666666666,15120.0,0.0


### Select candidates at or below 80 mph

In [ ]:
taxi_olcum_temiz = taxi_olcum.filter(
    F.col("ortalama_hiz_mph") <= 80
)

print("Ölçüm analizine seçilen kayıt:", taxi_olcum_temiz.count())

Ölçüm analizine seçilen kayıt: 3380592


### Build the Silver eligibility flag

In [ ]:
taxi_silver = (
    taxi_kalite
    .withColumn(
        "ortalama_hiz_mph",
        F.when(
            F.col("sure_dakika") > 0,
            F.col("trip_distance") * 60 / F.col("sure_dakika")
        )
    )
    .withColumn(
        "olcum_analizine_uygun",
        F.coalesce(
            (F.col("trip_distance") > 0) &
            (F.col("trip_distance") <= 100) &
            (F.col("sure_dakika") > 0) &
            (F.col("sure_dakika") <= 180) &
            (F.col("ortalama_hiz_mph") <= 80),
            F.lit(False)
        )
    )
)

display(
    taxi_silver
    .groupBy("olcum_analizine_uygun")
    .count()
)

olcum_analizine_uygun,count
false,94612
true,3380592


### Persist the Silver Delta table

In [ ]:
taxi_silver.createOrReplaceTempView("taxi_silver_hazir")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.nyc_taxi_silver_2025_01
    USING DELTA
    AS SELECT * FROM taxi_silver_hazir
""")

### Review persisted eligibility counts

In [ ]:
display(
    spark.table("workspace.default.nyc_taxi_silver_2025_01")
    .groupBy("olcum_analizine_uygun")
    .count()
)

olcum_analizine_uygun,count
false,94612
true,3380592


## Checks and recorded baseline
| Measure | Recorded value |
|---|---:|
| Raw-file records | 3,475,226 |
| Outside January pickup window | 22 |
| January / Silver records | 3,475,204 |
| Distance and duration candidates | 3,381,449 |
| Candidate speeds above 80 mph | 857 |
| Measurement-eligible records | 3,380,592 |
| Measurement-ineligible records retained in Silver | 94,612 |

The largest candidate speed is approximately 67,608 mph: 18.78 miles in one second. This is an implausible derived observation, not an actual measured vehicle speed.

Null and negative distance labels are explicit in the packaged version. The final eligibility predicate remains the original project rule: distance > 0 and <= 100 miles, duration > 0 and <= 180 minutes, speed <= 80 mph, and null predicates mapped to false.

## Next steps
Run notebook 03 for Silver analysis and notebook 04 to create the Gold day/hour table.